In [69]:
%pip install spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 20.0 MB/s eta 0:00:00MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.7/656.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.1/788.1 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [spacy]m━━ 14/15 [spacy][thinc]
Note: you may need to restart the kernel to use updated packages.


In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import spacy
from spacy.matcher import PhraseMatcher

In [60]:
df = pd.read_csv('df_hh.csv')
df

,title,hh_link,address,company,short_description,experience,salary
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки"
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN
...,...,...,...,...,...,...,...
13546,Бизнес-аналитик / Системный аналитик (Fullstac...,https://hh.ru/vacancy/131129287?query=%D0%BF%D...,Санкт-Петербург,ООО Апартмент системс,Опыт работы бизнес-аналитиком / системным анал...,Опыт 3-6 лет,NaN
13547,Девелопер обуви и аксессуаров,https://hh.ru/vacancy/131335836?query=%D0%BF%D...,"Москва, р-н Преображенское",Ushatava,"Аналогичный опыт работы от 3 лет, опыт работы",Опыт 1-3 года,NaN
13548,Product Analyst,https://hh.ru/vacancy/131303603?query=%D0%BF%D...,"Москва, р-н Раменки",Clear Mind,Опытом самостоятельного создания воронок и фор...,Опыт 3-6 лет,NaN
13549,Начальник отдела маркетинга,https://hh.ru/vacancy/128851476?query=%D0%BF%D...,"Челябинск, р-н Центральный",ООО Чебаркульская птица,"Опыт работы в маркетинге от 3 лет, из них не м...",Опыт 3-6 лет,NaN


In [61]:
df.describe()

,title,hh_link,address,company,short_description,experience,salary
count,13551,13551,13550,13550,13413,13550,3409
unique,7981,12135,1549,6170,10192,4,1072
top,Инженер по качеству,https://hh.ru/vacancy/131918299?query=it&hhtmF...,Москва,Сбер. IT,Опыт работы,Опыт 1-3 года,"от 100 000 ₽ за месяц, на руки"
freq,191,3,6492,360,183,6005,107


In [62]:
# очищаем дубли

df = df.drop_duplicates()
len(df)

12136

In [65]:
# Поработаем с графой опыта

df['short_description']

0                                              Опыт работы
1        Уверенное знание Python 3, базовых принципов О...
2        2+ года коммерческой разработки на PHP 7.4+. У...
3                  Опыт работы с АБС «ЦФТ-Банк» в качестве
4        Опыт разработки или исправления/доработки внут...
                               ...                        
13546    Опыт работы бизнес-аналитиком / системным анал...
13547        Аналогичный опыт работы от 3 лет, опыт работы
13548    Опытом самостоятельного создания воронок и фор...
13549    Опыт работы в маркетинге от 3 лет, из них не м...
13550    Опыт работы с CJ, вовлекающими механиками, ист...
Name: short_description, Length: 12136, dtype: object

Нам нужно достать навыки из описания вакансии

Нашел [гитхаб](https://github.com/iconicompany/iskillmatching) человечка, который уже решал такую проблему. Адаптируем такое решение

Пайплайн будет следующий:
1. Извлекаем по шаблонам навыки
2. Нормализуем навыки для удобного анализа. Например Python3 -> Python

In [ ]:
# очень удобно воруем) 

skills = pd.read_csv('skills.csv') 
skills

,Название
0,*nix
1,-
2,.NET
3,.NET 3
4,".net 3,5"
...,...
15854,Яндекс.Вики
15855,Яндекс.Касса
15856,Яндекс.Метрика
15857,Яндекс.Трекер


In [74]:
spacy.cli.download("ru_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 25.7 MB/s eta 0:00:00 MB/s eta 0:00:0101
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 5.4 MB/s eta 0:00:00 MB/s eta 0:00:01:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ru-core-news-sm]3/4 [ru-core-news-sm]
✔ Download and installation successful
You can now load the package via spacy.load('ru_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
# Для извлечения заиспользуем PhraseMatcher из SpaCy.
# Он быстро работает на большом объеме данных


def get_spacy_matcher(nlp, skills_list):
    matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
    patterns = [nlp.make_doc(skill) for skill in skills_list]
    matcher.add("SKILL", patterns)
    return matcher

def extract_skills_from_doc(doc, matcher):
    seen = set()
    out = []
    for _match_id, start, end in matcher(doc):
        key = (start, end)
        if key in seen:
            continue
        seen.add(key)
        out.append(doc[start:end].text.strip())

    return out


nlp = spacy.load("ru_core_news_sm")


def get_skills_list(skills):
    # skills — это DataFrame; в матчер нужен список строк из колонки «Название»
    s = skills["Название"].dropna().astype(str).str.strip()
    return s.unique().tolist()

skills_list = get_skills_list(skills)
matcher = get_spacy_matcher(nlp, skills_list)

def extract_skills_text(text):
    if not isinstance(text, str) or not text.strip():
        return []
    doc = nlp(text)
    return extract_skills_from_doc(doc, matcher)

In [ ]:
# применяем 

df["skills_matched"] = df["short_description"].fillna("").map(extract_skills_text)

df

/var/folders/v8/lqntntc15fgctlxw4wdhwhx80000gn/T/ipykernel_33751/3357000340.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["skills_matched"] = df["short_description"].fillna("").map(extract_skills_text)


,title,hh_link,address,company,short_description,experience,salary,experience_min_years,skills_matched
0,Разработчик,https://hh.ru/vacancy/131819443?query=%D1%80%D...,Москва,"Русская Медиагруппа, радиохолдинг",Опыт работы,Опыт 3-6 лет,NaN,3.0,[]
1,Разработчик,https://hh.ru/vacancy/131986214?query=%D1%80%D...,Москва,Детский хоспис Дом с маяком,"Уверенное знание Python 3, базовых принципов О...",Опыт 1-3 года,"от 95 000 ₽ за месяц, на руки",1.0,"[знание, Python, Понимание, данных, SQL, JSON,..."
2,Backend-разработчик,https://hh.ru/vacancy/131274910?query=%D1%80%D...,Москва,ООО ГК СтиС,2+ года коммерческой разработки на PHP 7.4+. У...,Опыт 1-3 года,NaN,1.0,"[PHP, PHP 7.4, PHP 7.4+, Laravel, Laravel 7, E..."
3,Разработчик ЦФТ,https://hh.ru/vacancy/132003073?query=%D1%80%D...,Москва,Bell Integrator,Опыт работы с АБС «ЦФТ-Банк» в качестве,Опыт 3-6 лет,NaN,3.0,"[АБС, ЦФТ, ЦФТ-Банк, Банк]"
4,Разработчик C++,https://hh.ru/vacancy/132001730?query=%D1%80%D...,Москва,ООО БУЛАТ,Опыт разработки или исправления/доработки внут...,Опыт 3-6 лет,NaN,3.0,"[Linux, Kernel, Linux, оптимизация]"
...,...,...,...,...,...,...,...,...,...
13546,Бизнес-аналитик / Системный аналитик (Fullstac...,https://hh.ru/vacancy/131129287?query=%D0%BF%D...,Санкт-Петербург,ООО Апартмент системс,Опыт работы бизнес-аналитиком / системным анал...,Опыт 3-6 лет,NaN,3.0,"[бизнес, IT]"
13547,Девелопер обуви и аксессуаров,https://hh.ru/vacancy/131335836?query=%D0%BF%D...,"Москва, р-н Преображенское",Ushatava,"Аналогичный опыт работы от 3 лет, опыт работы",Опыт 1-3 года,NaN,1.0,[]
13548,Product Analyst,https://hh.ru/vacancy/131303603?query=%D0%BF%D...,"Москва, р-н Раменки",Clear Mind,Опытом самостоятельного создания воронок и фор...,Опыт 3-6 лет,NaN,3.0,"[web, SQL, Python, pandas, numpy, matplotlib]"
13549,Начальник отдела маркетинга,https://hh.ru/vacancy/128851476?query=%D0%BF%D...,"Челябинск, р-н Центральный",ООО Чебаркульская птица,"Опыт работы в маркетинге от 3 лет, из них не м...",Опыт 3-6 лет,NaN,3.0,[]
